# 🔲 Module 2.3: Morphological Operations — Shape-Based Image Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_02_Image_Processing_Basics/03_Morphological_Operations/morphological_operations.ipynb)

---

## 🎯 Learning Objectives
1. Erosion, dilation — set theory foundation
2. Opening, closing — compound operations
3. Hit-or-miss transform, morphological gradient
4. Morphological operations as RL actions for shape processing

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless -q

import numpy as np
import matplotlib.pyplot as plt
import cv2

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision

# CIFAR-10: 60,000 real photographs in 10 classes (auto-downloads ~170MB)
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10 loaded: {len(cifar10)} real photos (32x32x3)")
print(f"   Classes: {cifar10.classes}")

# scikit-image real test images
from skimage import data
sample_images = {
    'astronaut': data.astronaut(),
    'camera': data.camera(), 
    'coins': data.coins(),
    'horse': data.horse(),
}
print(f"✅ scikit-image: {len(sample_images)} real test images loaded")

## Deep Mathematical Derivation: Morphological Operations from Set Theory

### Step 1: Binary Image as a Set
A binary image $A$ is a subset of $\mathbb{Z}^2$:
$$A = \{(x, y) \in \mathbb{Z}^2 : I(x, y) = 1\}$$

The structuring element $B$ is also a set, centered at origin.

### Step 2: Erosion (Formal Definition + Derivation)
$$A \ominus B = \{z \in \mathbb{Z}^2 : B_z \subseteq A\}$$

where $B_z = \{b + z : b \in B\}$ is $B$ translated to position $z$.

**In words:** A pixel $z$ survives erosion iff the ENTIRE structuring element fits inside $A$ when centered at $z$.

**Proof that erosion shrinks objects:**
If $z \in A \ominus B$, then $B_z \subseteq A$, so $z + b \in A$ for all $b \in B$. Since $\mathbf{0} \in B$ (origin is in structuring element), $z \in A$. Therefore $A \ominus B \subseteq A$. $\blacksquare$

### Step 3: Dilation (Formal Definition + Derivation)
$$A \oplus B = \{z \in \mathbb{Z}^2 : (\hat{B})_z \cap A \neq \emptyset\}$$

where $\hat{B} = \{-b : b \in B\}$ is the reflection of $B$.

**Equivalently:** $A \oplus B = \bigcup_{b \in B} A_b = \{a + b : a \in A, b \in B\}$ (Minkowski sum)

**Proof that dilation expands objects:**
For any $a \in A$, since $\mathbf{0} \in B$, we have $a + \mathbf{0} = a \in A \oplus B$. Therefore $A \subseteq A \oplus B$. $\blacksquare$

### Step 4: Duality Theorem (Erosion ↔ Dilation)
**Theorem:** $(A \ominus B)^c = A^c \oplus \hat{B}$

**Proof:**
$$z \notin A \ominus B \iff B_z \not\subseteq A \iff \exists b \in B : z + b \notin A$$
$$\iff \exists b \in B : z + b \in A^c \iff (\hat{B})_z \cap A^c \neq \emptyset \iff z \in A^c \oplus \hat{B} \quad \blacksquare$$

**Consequence:** Erosion of foreground = dilation of background (and vice versa).

### Step 5: Opening and Closing — Derivation of Properties

**Opening:** $A \circ B = (A \ominus B) \oplus B$

**Closing:** $A \bullet B = (A \oplus B) \ominus B$

**Proof that opening is anti-extensive:** $A \circ B \subseteq A$

From Step 2: $A \ominus B \subseteq A$. For any $z \in (A \ominus B) \oplus B$, there exists $a \in A \ominus B$ and $b \in B$ with $z = a + b$. Since $a \in A \ominus B$, $B_a \subseteq A$, so $a + b \in A$, i.e., $z \in A$. $\blacksquare$

**Proof that closing is extensive:** $A \subseteq A \bullet B$

For any $a \in A$, $a \in A \oplus B$ (since $\mathbf{0} \in B$). Now consider $a$ in the erosion step: $B_a = \{a + b : b \in B\}$. Since $a \in A$ and $a \in A \oplus B$, we need $B_a \subseteq A \oplus B$. For any $b \in B$, $a + b \in A \oplus B$ (by definition of dilation). Therefore $a \in (A \oplus B) \ominus B$. $\blacksquare$

**Proof of idempotence:** $(A \circ B) \circ B = A \circ B$

Opening removes all parts of $A$ that don't fit $B$. Applying opening again doesn't remove anything further because the remaining structure already accommodates $B$. (Formal proof via the fact that $A \circ B$ is the union of all translates of $B$ that fit inside $A$.) $\blacksquare$

### Step 6: Morphological Gradient
$$\text{grad}(A) = (A \oplus B) - (A \ominus B)$$

**Proof that this detects boundaries:** Dilation expands outward, erosion shrinks inward. Their difference captures exactly the boundary region of width determined by $B$. $\blacksquare$

### RL Connection: Morphological Operations as Actions
For binary segmentation RL, the action space includes morphological operations:
$$\mathcal{A} = \{\text{erode}(B_1), \text{dilate}(B_1), \text{open}(B_2), \text{close}(B_2), \text{identity}\}$$

The agent learns to clean up segmentation masks by choosing the right sequence of operations — smoothing boundaries (opening), filling holes (closing), or removing noise (erosion).

## Algebraic Properties of Morphological Operations — Formal Proofs

Morphological operations form a rich algebraic structure. Understanding these properties is essential for designing optimal sequences of operations (which is exactly what an RL agent must learn).

### Property 1: Associativity of Dilation

**Theorem:** $(A \oplus B) \oplus C = A \oplus (B \oplus C)$

**Proof:**
$$z \in (A \oplus B) \oplus C \iff \exists c \in C : z - c \in A \oplus B$$
$$\iff \exists c \in C, \exists b \in B : z - c - b \in A$$
$$\iff \exists d \in B \oplus C : z - d \in A \quad (\text{where } d = b + c)$$
$$\iff z \in A \oplus (B \oplus C) \quad \blacksquare$$

### Property 2: Distributivity of Dilation over Union

**Theorem:** $A \oplus (B \cup C) = (A \oplus B) \cup (A \oplus C)$

**Proof:**
$$z \in A \oplus (B \cup C) \iff \exists x \in A, \exists y \in B \cup C : z = x + y$$
$$\iff (\exists x \in A, \exists y \in B : z = x + y) \text{ or } (\exists x \in A, \exists y \in C : z = x + y)$$
$$\iff z \in (A \oplus B) \text{ or } z \in (A \oplus C) \iff z \in (A \oplus B) \cup (A \oplus C) \quad \blacksquare$$

### Property 3: Translation Invariance

**Theorem:** $(A_t) \oplus B = (A \oplus B)_t$ where $A_t = \{a + t : a \in A\}$

**Proof:**
$$z \in (A_t) \oplus B \iff \exists a' \in A_t, b \in B : z = a' + b$$
$$\iff \exists a \in A, b \in B : z = (a + t) + b = (a + b) + t$$
$$\iff z - t \in A \oplus B \iff z \in (A \oplus B)_t \quad \blacksquare$$

### Property 4: Monotonicity (Increasing Property)

**Theorem:** If $A_1 \subseteq A_2$, then $A_1 \oplus B \subseteq A_2 \oplus B$ and $A_1 \ominus B \subseteq A_2 \ominus B$.

**Proof (dilation):** Let $z \in A_1 \oplus B$. Then $\exists a \in A_1, b \in B : z = a + b$. Since $A_1 \subseteq A_2$, $a \in A_2$, so $z \in A_2 \oplus B$. $\blacksquare$

**Proof (erosion):** Let $z \in A_1 \ominus B$. Then $B_z \subseteq A_1 \subseteq A_2$, so $B_z \subseteq A_2$, meaning $z \in A_2 \ominus B$. $\blacksquare$

### Property 5: Decomposition of Structuring Elements

**Theorem:** If $B = B_1 \oplus B_2$, then $A \oplus B = (A \oplus B_1) \oplus B_2$

This is crucial for computational efficiency: a large disk SE can be decomposed into repeated dilations with a small SE, reducing $O(N^2 |B|)$ to $O(N^2 \cdot k \cdot |B_{\text{small}}|)$.

### RL Implication: Action Composition

These algebraic properties tell the RL agent that:
- **Associativity** → the agent can plan multi-step morphological strategies
- **Distributivity** → processing regions independently then merging is valid
- **Decomposition** → complex operations can be built from simpler primitives
- The agent's action space has a **group-like structure** that can be exploited for more efficient exploration

## Hit-or-Miss Transform — Template Matching via Morphology

The hit-or-miss transform is the most powerful morphological tool for detecting specific patterns. It extends erosion to simultaneously match both foreground AND background.

### Step 1: Composite Structuring Element

Define a composite SE as a pair $(B_1, B_2)$ where:
- $B_1$: the "hit" part — pixels that must be in the foreground (1)
- $B_2$: the "miss" part — pixels that must be in the background (0)
- $B_1 \cap B_2 = \emptyset$ (no overlap allowed)

**Example — detecting top-left corners:**
$$B_1 = \begin{bmatrix} 0 & 0 & 0 \\ 0 & 1 & 1 \\ 0 & 1 & 0 \end{bmatrix}, \quad B_2 = \begin{bmatrix} 1 & 1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix}$$

### Step 2: Formal Definition

$$A \circledast (B_1, B_2) = (A \ominus B_1) \cap (A^c \ominus B_2)$$

**In words:** Find positions where $B_1$ fits inside $A$ AND $B_2$ fits inside $A^c$ (complement) simultaneously.

### Step 3: Proof of Equivalence to Template Matching

**Theorem:** $z \in A \circledast (B_1, B_2)$ if and only if the pattern defined by $(B_1, B_2)$ exactly matches the image at position $z$.

**Proof:**
$$z \in A \circledast (B_1, B_2) \iff z \in (A \ominus B_1) \text{ AND } z \in (A^c \ominus B_2)$$
$$\iff (B_1)_z \subseteq A \text{ AND } (B_2)_z \subseteq A^c$$
$$\iff \forall b_1 \in B_1: z + b_1 \in A \text{ AND } \forall b_2 \in B_2: z + b_2 \notin A$$

This means: all "hit" pixels are foreground, and all "miss" pixels are background — an exact template match. $\blacksquare$

### Step 4: Morphological Thinning and Thickening

The hit-or-miss transform enables iterative shape refinement:

**Thinning:** $A \oslash (B_1, B_2) = A \setminus [A \circledast (B_1, B_2)] = A \cap [A \circledast (B_1, B_2)]^c$

Repeatedly applying thinning with rotated SEs extracts the **skeleton** of a shape — the minimal set of curves that preserves topology.

**Thickening:** $A \odot (B_1, B_2) = A \cup [A \circledast (B_1, B_2)]$

### Step 5: Skeletonization via Hit-or-Miss

The morphological skeleton $S(A)$ is computed by iterative thinning until convergence:

$$S(A) = A \oslash (B_1^{(1)}, B_2^{(1)}) \oslash (B_1^{(2)}, B_2^{(2)}) \oslash \cdots$$

where $(B_1^{(i)}, B_2^{(i)})$ cycle through 8 rotations of the thinning SE.

**Theorem:** The skeleton preserves the topology (connected components, holes) of the original shape.

### RL Application

An RL agent for shape analysis can use the hit-or-miss transform as a **feature detector**: different composite SEs detect different structural features (corners, endpoints, T-junctions), providing the agent with a rich, geometrically meaningful feature set for decision-making.

## 1. Mathematical Foundation — Set Theory

Binary images are sets: $A \subseteq \mathbb{Z}^2$ where $A$ contains coordinates of white pixels.

**Structuring Element (SE):** $B \subseteq \mathbb{Z}^2$ — a small shape used as a probe.

### Erosion:
$$A \ominus B = \{z \in \mathbb{Z}^2 : B_z \subseteq A\}$$

"Places where B fits entirely inside A" — **shrinks** objects.

### Dilation:
$$A \oplus B = \{z \in \mathbb{Z}^2 : (\hat{B})_z \cap A \neq \emptyset\}$$

"Places where B overlaps with A" — **grows** objects.

### Properties:
- **Duality:** $(A \ominus B)^c = A^c \oplus \hat{B}$
- **Associative:** $A \oplus (B \oplus C) = (A \oplus B) \oplus C$
- **Increasing:** If $A \subseteq C$, then $A \oplus B \subseteq C \oplus B$

### Compound Operations:
- **Opening:** $A \circ B = (A \ominus B) \oplus B$ — removes small protrusions
- **Closing:** $A \bullet B = (A \oplus B) \ominus B$ — fills small holes
- **Morphological Gradient:** $(A \oplus B) - (A \ominus B)$ — edge detection!

## Structuring Element Design — Shape, Size, and Decomposition

The choice of structuring element determines the behavior of every morphological operation. Understanding SE design is crucial for an RL agent that selects morphological actions.

### Step 1: Common Structuring Elements

**Square SE** (side $s$): $B_{\text{square}} = \{(i,j) : |i| \leq s/2, |j| \leq s/2\}$

**Disk SE** (radius $r$): $B_{\text{disk}} = \{(i,j) : i^2 + j^2 \leq r^2\}$

**Diamond SE** (radius $r$): $B_{\text{diamond}} = \{(i,j) : |i| + |j| \leq r\}$

**Line SE** (length $l$, angle $\theta$): $B_{\text{line}} = \{(t\cos\theta, t\sin\theta) : |t| \leq l/2\}$

### Step 2: Decomposition for Efficiency

**Theorem:** A large square SE of side $2n+1$ can be decomposed as $n$ dilations with a $3 \times 3$ square:

$$B_{2n+1} = \underbrace{B_3 \oplus B_3 \oplus \cdots \oplus B_3}_{n \text{ times}}$$

**Proof:** $B_3 = \{-1, 0, 1\}^2$. The Minkowski sum $B_3 \oplus B_3 = \{-2, -1, 0, 1, 2\}^2 = B_5$. By induction: $B_3^{\oplus n} = B_{2n+1}$. $\blacksquare$

**Computational savings:** Dilation with a $21 \times 21$ SE: $O(N^2 \cdot 441)$ operations. Decomposed as 10 dilations with $3 \times 3$: $O(N^2 \cdot 10 \cdot 9) = O(N^2 \cdot 90)$ — a **4.9× speedup**.

### Step 3: Disk Approximation by Polygon Decomposition

A disk SE can be approximated by the union of line SEs at multiple angles:

$$B_{\text{disk}} \approx B_{\text{line}}^{0°} \cup B_{\text{line}}^{45°} \cup B_{\text{line}}^{90°} \cup B_{\text{line}}^{135°}$$

Or more efficiently: decompose using a sequence of $3 \times 3$ kernels with specific shapes, alternating between different orientations. The hexagonal decomposition of Adams (1993) achieves disk erosion/dilation in $O(r)$ operations per pixel instead of $O(r^2)$.

### Step 4: Anisotropic Structuring Elements

Directional SEs detect features along specific orientations:

**Example:** Horizontal line SE $\begin{bmatrix} 0 & 0 & 0 \\ 1 & 1 & 1 \\ 0 & 0 & 0 \end{bmatrix}$ detects horizontal structures.

**Opening with rotated SEs** at angles $\theta_1, \ldots, \theta_K$:
$$\gamma_{\text{directional}}(A) = \bigcup_{k=1}^{K} (A \circ B_{\theta_k})$$

This extracts features at any of the $K$ orientations — useful for detecting blood vessels, road networks, or cracks.

### RL Action Space Design

An RL agent can parameterize the SE:
$$\mathcal{A}_{\text{SE}} = \{(\text{shape}, \text{size}, \text{angle})\} = \{\text{square}, \text{disk}, \text{diamond}\} \times \{3, 5, 7, 11\} \times \{0°, 45°, 90°, 135°\}$$

This gives $3 \times 4 \times 4 = 48$ possible SEs per morphological operation — a manageable discrete action space that covers the important design choices.

In [ ]:
# Create a binary image with various features
size = 128
img = np.zeros((size, size), dtype=np.uint8)

# Main shape
img[20:100, 20:100] = 255

# Add small protrusions (to be removed by opening)
img[10:15, 50:55] = 255
img[105:110, 70:75] = 255
img[50:55, 105:110] = 255

# Add holes (to be filled by closing)
img[40:50, 40:50] = 0
img[60:68, 60:68] = 0
img[75:80, 30:35] = 0

# Add noise
noise = np.random.randint(0, 2, (size, size)).astype(np.uint8) * 255
noise_mask = np.random.random((size, size)) < 0.02
img[noise_mask] = noise[noise_mask]

# Structuring elements
kernel_3x3 = np.ones((3, 3), dtype=np.uint8)
kernel_5x5 = np.ones((5, 5), dtype=np.uint8)
kernel_cross = cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5))
kernel_ellipse = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

# Apply operations
eroded = cv2.erode(img, kernel_3x3, iterations=1)
dilated = cv2.dilate(img, kernel_3x3, iterations=1)
opened = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel_3x3)
closed = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel_3x3)
gradient = cv2.morphologyEx(img, cv2.MORPH_GRADIENT, kernel_3x3)
tophat = cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel_5x5)

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

results = [
    (img, 'Original'),
    (eroded, 'Erosion (A⊖B)\nShrinks objects'),
    (dilated, 'Dilation (A⊕B)\nGrows objects'),
    (gradient, 'Gradient\n(A⊕B)-(A⊖B)'),
    (opened, 'Opening (A∘B)\nRemoves protrusions'),
    (closed, 'Closing (A•B)\nFills holes'),
    (tophat, 'Top Hat\nA-(A∘B)'),
    (opened, 'Open then Close\n(Clean version)'),
]

# Apply open+close for last
clean = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel_3x3)
results[7] = (clean, 'Open→Close\n(Fully cleaned)')

for idx, (result, title) in enumerate(results):
    row, col = idx // 4, idx % 4
    axes[row, col].imshow(result, cmap='gray')
    axes[row, col].set_title(title, fontsize=10)
    axes[row, col].axis('off')

plt.suptitle('Morphological Operations — Shape Processing Toolkit\n'
             'Each operation = one RL action for cleaning/refining images!',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('morphological_ops.png', dpi=150, bbox_inches='tight')
plt.show()

## Distance Transform — Euclidean Distance from Morphological Operations

The distance transform assigns to each pixel the distance to the nearest boundary, connecting morphology to metric geometry.

### Step 1: Definition

The Euclidean distance transform of a binary image $A$:

$$D_A(\mathbf{x}) = \min_{\mathbf{y} \in \partial A} \|\mathbf{x} - \mathbf{y}\|_2$$

where $\partial A$ is the boundary of set $A$.

### Step 2: Connection to Erosion

**Theorem:** The distance transform can be computed via iterated erosions:

$$D_A(\mathbf{x}) = \max\{r \geq 0 : \mathbf{x} \in A \ominus rB\}$$

where $rB$ is a disk of radius $r$.

**Proof:** $\mathbf{x} \in A \ominus rB$ iff the disk $rB$ centered at $\mathbf{x}$ fits entirely inside $A$, which means $\|\mathbf{x} - \mathbf{y}\|_2 \leq r$ for all $\mathbf{y} \in (rB)_\mathbf{x}$, and all such $\mathbf{y}$ are in $A$. The maximum such $r$ equals the distance to the nearest boundary point. $\blacksquare$

### Step 3: Efficient Computation (Two-Pass Algorithm)

**Squared EDT via separable computation** (Saito-Toriwaki, 1994):

Pass 1 (horizontal): For each row, compute 1D distance to nearest background pixel
Pass 2 (vertical): For each column, compute the minimum over rows

$$D^2(\mathbf{x}) = \min_j \{D_1^2(x, j) + (y - j)^2\}$$

This computes the exact Euclidean distance transform in $O(N)$ time (linear in image size), compared to $O(N \cdot |\partial A|)$ for the naive approach.

### Step 4: Applications of the Distance Transform

**1. Watershed segmentation:** Use $-D_A$ as a topographic surface; watersheds divide touching objects.

**2. Voronoi diagram:** The distance transform implicitly encodes the nearest-feature Voronoi partition:
$$\text{Voronoi cell of } \mathbf{y}_i = \{\mathbf{x} : D_{A}(\mathbf{x}) = \|\mathbf{x} - \mathbf{y}_i\|\}$$

**3. Skeletonization:** The skeleton (medial axis) consists of points equidistant from two or more boundary points — the ridges of $D_A$.

### Step 5: RL Reward Shaping via Distance Transform

For an RL segmentation agent, the distance transform provides a natural reward shaping function:

$$R_{\text{shaped}}(\mathbf{x}) = \begin{cases} -D_{\text{target}}(\mathbf{x}) & \text{if } \mathbf{x} \text{ is a false negative} \\ -D_{\text{predicted}}(\mathbf{x}) & \text{if } \mathbf{x} \text{ is a false positive} \\ 0 & \text{if correct} \end{cases}$$

This penalizes errors proportionally to their distance from the correct boundary, providing a smoother reward landscape than binary IoU — enabling faster policy gradient learning.

## Granulometry — Size Distribution Theory

Granulometry uses morphological operations to analyze the size distribution of objects in an image, analogous to sieve analysis in geology.

### Step 1: Morphological Opening with Scaled Structuring Elements

Define a family of openings with increasing structuring element size:

$$\gamma_r(A) = A \circ rB$$

where $rB = \{rb : b \in B\}$ is the structuring element $B$ scaled by factor $r$.

### Step 2: Pattern Spectrum (Size Distribution)

The **pattern spectrum** $\text{PS}(r)$ measures how much "area" (number of foreground pixels) is removed at each scale:

$$\Omega(r) = \text{Area}(\gamma_r(A)) = |\gamma_r(A)|$$

$$\text{PS}(r) = -\frac{d\Omega}{dr} = \Omega(r) - \Omega(r + \delta r)$$

**Properties:**

1. $\Omega(r)$ is monotonically non-increasing (opening removes more as $r$ increases)
2. $\Omega(0) = |A|$ (identity: opening with point SE = original)
3. $\Omega(\infty) = 0$ (eventually everything is removed)
4. $\text{PS}(r) \geq 0$ (area can only decrease with larger openings)

### Step 3: Proof of Monotonicity

**Theorem:** If $r_1 < r_2$, then $\gamma_{r_2}(A) \subseteq \gamma_{r_1}(A)$.

**Proof:** Since $r_1 B \subseteq r_2 B$ for convex $B$ and $r_1 < r_2$:
$$A \ominus r_2 B \subseteq A \ominus r_1 B$$
(erosion with a larger SE removes more, by the increasing property). Therefore:
$$(A \ominus r_2 B) \oplus r_2 B \subseteq (A \ominus r_1 B) \oplus r_1 B$$
since dilation is increasing in both arguments. $\blacksquare$

### Step 4: Interpreting the Pattern Spectrum

The pattern spectrum $\text{PS}(r)$ tells us the distribution of object sizes:
- A peak at $r = r_0$ means many objects have characteristic size $\approx r_0$
- The width of the peak indicates size variability
- Multiple peaks indicate a multi-modal size distribution

**Moments of the pattern spectrum:**
$$\bar{r} = \frac{\sum_r r \cdot \text{PS}(r)}{\sum_r \text{PS}(r)} \quad (\text{mean size})$$

$$\sigma_r^2 = \frac{\sum_r (r - \bar{r})^2 \cdot \text{PS}(r)}{\sum_r \text{PS}(r)} \quad (\text{size variance})$$

### Step 5: Application to RL for Quality Control

In an industrial RL setting, the agent can use granulometry features as part of its state:
$$\mathbf{s}_t = [\bar{r}, \sigma_r, \text{PS}(r_1), \ldots, \text{PS}(r_K)]$$

This compact representation captures the essential size distribution of objects, enabling the agent to make decisions about manufacturing quality (e.g., reject parts with objects too large or too small).

## Gray-Scale Morphology — Extension from Binary to Continuous Images

Binary morphology operates on sets. Extending to gray-scale images requires replacing set operations with min/max operations on intensity surfaces.

### Step 1: Gray-Scale Dilation

For a gray-scale image $f(x, y)$ and a flat structuring element $B$:

$$(f \oplus B)(x, y) = \max_{(s,t) \in B} f(x - s, y - t)$$

**Interpretation:** Slide $B$ over the image; at each position, take the maximum pixel value in the neighborhood defined by $B$. This brightens the image and expands bright regions.

For a **non-flat** (grayscale) structuring element $b(s, t)$:

$$(f \oplus b)(x, y) = \max_{(s,t) \in B} [f(x - s, y - t) + b(s, t)]$$

### Step 2: Gray-Scale Erosion

$$(f \ominus B)(x, y) = \min_{(s,t) \in B} f(x + s, y + t)$$

**Interpretation:** At each position, take the minimum pixel value. This darkens the image and shrinks bright regions.

### Step 3: Proof of Duality in Gray-Scale

**Theorem:** $(f \ominus B)^c = f^c \oplus \hat{B}$ where $f^c(x,y) = -f(x,y)$ (complement = negation).

**Proof:**
$$(f \ominus B)^c(x,y) = -\min_{(s,t) \in B} f(x+s, y+t) = \max_{(s,t) \in B} [-f(x+s, y+t)]$$
$$= \max_{(s,t) \in B} f^c(x+s, y+t) = (f^c \oplus \hat{B})(x,y) \quad \blacksquare$$

### Step 4: Morphological Gradient for Edge Detection

$$\text{grad}(f) = (f \oplus B) - (f \ominus B)$$

**Physical meaning:** Dilation gives the local maximum intensity, erosion gives the local minimum. Their difference measures the local contrast — large values indicate edges.

**Internal gradient:** $\text{grad}_{\text{int}}(f) = f - (f \ominus B)$ (highlights the inside boundary)

**External gradient:** $\text{grad}_{\text{ext}}(f) = (f \oplus B) - f$ (highlights the outside boundary)

### Step 5: Top-Hat and Bottom-Hat Transforms

**White top-hat** (extracts bright details smaller than $B$):
$$\text{WTH}(f) = f - (f \circ B)$$

**Black top-hat** (extracts dark details smaller than $B$):
$$\text{BTH}(f) = (f \bullet B) - f$$

**Proof that WTH extracts small bright objects:**
Opening $f \circ B$ removes bright features smaller than $B$. Subtracting the opened image from the original recovers exactly those removed features. $\blacksquare$

### Step 6: Contrast Enhancement via Morphology

$$f_{\text{enhanced}} = f + \text{WTH}(f) - \text{BTH}(f)$$

This simultaneously brightens small bright features and darkens small dark features, enhancing local contrast — a useful action for an RL image enhancement agent.

## Morphological Reconstruction — Geodesic Operations

Morphological reconstruction is a powerful technique that extracts connected components based on a marker image. Unlike basic erosion/dilation, reconstruction preserves the exact shape of objects.

### Step 1: Geodesic Dilation

**Definition:** The geodesic dilation of marker $J$ under mask $I$ (with $J \leq I$ pointwise):

$$\delta_I^{(1)}(J) = \min(J \oplus B, I)$$

The mask $I$ constrains the dilation — the result never exceeds the mask.

**Iterated geodesic dilation:**
$$\delta_I^{(n)}(J) = \underbrace{\delta_I^{(1)}(\delta_I^{(1)}(\cdots \delta_I^{(1)}}_{n \text{ times}}(J)\cdots))$$

### Step 2: Reconstruction by Dilation

**Definition:** The reconstruction of mask $I$ from marker $J$ is the geodesic dilation iterated until convergence:

$$R_I(J) = \delta_I^{(\infty)}(J) = \lim_{n \to \infty} \delta_I^{(n)}(J)$$

**Proof of convergence:** The sequence $\{\delta_I^{(n)}(J)\}$ is:
1. Monotonically non-decreasing: $\delta_I^{(n)}(J) \leq \delta_I^{(n+1)}(J)$ (dilation is extensive)
2. Bounded above by $I$: $\delta_I^{(n)}(J) \leq I$ for all $n$ (by the min with $I$)

A bounded monotone sequence converges. $\blacksquare$

### Step 3: Properties of Reconstruction

1. **Idempotent:** $R_I(R_I(J)) = R_I(J)$
2. **Anti-extensive:** $R_I(J) \leq I$
3. **Increasing:** If $J_1 \leq J_2$, then $R_I(J_1) \leq R_I(J_2)$
4. **Connected component extraction:** $R_I(J)$ extracts exactly the connected components of $I$ that contain at least one marker pixel from $J$

### Step 4: Opening by Reconstruction

$$\gamma_R(I) = R_I(I \ominus B)$$

**Advantage over standard opening:** Standard opening $(I \circ B)$ can alter the shape of objects. Opening by reconstruction preserves the exact contours of objects that survive erosion.

**Proof:** Erosion removes objects smaller than $B$. Reconstruction restores the surviving objects to their original shape (since the mask $I$ constrains the dilation to the original boundaries). $\blacksquare$

### Step 5: H-Maxima and H-Minima Transforms

**H-maxima:** Suppresses all maxima with height $\leq h$:
$$\text{HMAX}_h(I) = R_I(I - h)$$

**H-minima:** Suppresses all minima with depth $\leq h$:
$$\text{HMIN}_h(I) = [R_{I^c}(I^c - h)]^c$$

### Step 6: Application to RL — Robust Object Extraction

In an RL segmentation pipeline:
1. Agent places markers (seeds) on suspected objects
2. Reconstruction extracts complete connected objects from those seeds
3. Reward = IoU between reconstructed objects and ground truth

This gives the agent a powerful tool: instead of pixel-perfect segmentation, it only needs to approximately locate objects, and reconstruction handles the rest.

## Morphological Operations as a Complete Lattice — Abstract Algebraic Structure

The theory of morphological operations gains elegance and power when viewed through the lens of complete lattice theory.

### Step 1: The Lattice of Binary Images

The set of all binary images $\mathcal{P}(\mathbb{Z}^2)$ (power set of the integer grid) forms a **complete lattice** under set inclusion:

- **Partial order:** $A \leq B \iff A \subseteq B$
- **Join (supremum):** $A \vee B = A \cup B$
- **Meet (infimum):** $A \wedge B = A \cap B$
- **Least element:** $\emptyset$ (empty image)
- **Greatest element:** $\mathbb{Z}^2$ (all-white image)

### Step 2: Erosion and Dilation as Adjunctions

**Theorem:** Dilation $\delta_B$ and erosion $\varepsilon_B$ form an **adjunction** (Galois connection):

$$\delta_B(A) \leq C \iff A \leq \varepsilon_{\hat{B}}(C)$$

**Proof ($\Rightarrow$):** If $A \oplus B \subseteq C$, then for any $a \in A$: $\{a\} \oplus B \subseteq C$, so $B_a \subseteq C$, meaning $a \in C \ominus \hat{B}$. Therefore $A \subseteq C \ominus \hat{B}$. The reverse direction follows similarly. $\blacksquare$

### Step 3: Consequences of the Adjunction

From this single property, all major morphological identities follow:

1. **Dilation distributes over union:** $\delta_B(\bigcup_i A_i) = \bigcup_i \delta_B(A_i)$ (left adjoint preserves suprema)

2. **Erosion distributes over intersection:** $\varepsilon_B(\bigcap_i A_i) = \bigcap_i \varepsilon_B(A_i)$ (right adjoint preserves infima)

3. **Opening is a morphological filter:** $\gamma = \delta_B \circ \varepsilon_B$ is:
   - Increasing: $A \leq B \implies \gamma(A) \leq \gamma(B)$
   - Anti-extensive: $\gamma(A) \leq A$
   - Idempotent: $\gamma(\gamma(A)) = \gamma(A)$

4. **Closing is a morphological filter:** $\phi = \varepsilon_{\hat{B}} \circ \delta_B$ is increasing, extensive, and idempotent.

### Step 4: Morphological Filters as Projections

**Theorem:** Opening and closing are **projection operators** onto sublattices.

**Proof (opening):** We need to show: (a) idempotency and (b) the range is a sublattice.

(a) **Idempotency:** $\gamma(\gamma(A)) = \gamma(A)$

Since $\gamma(A) = (A \ominus B) \oplus B$, let $C = A \ominus B$. Then $\gamma(A) = C \oplus B$.

$\gamma(\gamma(A)) = ((C \oplus B) \ominus B) \oplus B$. Since $C \subseteq (C \oplus B) \ominus B$ (by adjunction) and $(C \oplus B) \ominus B \subseteq C$ (anti-extensivity of $\varepsilon \circ \delta$), we get $(C \oplus B) \ominus B = C$. Therefore $\gamma(\gamma(A)) = C \oplus B = \gamma(A)$. $\blacksquare$

(b) **Range is a sublattice:** The range of $\gamma$ consists of all unions of translates of $B$ — this is closed under union, forming a sublattice. $\blacksquare$

### Step 5: Significance for RL

The lattice structure means that morphological operations form a well-ordered algebra. For an RL agent, this translates to:
- **Guaranteed convergence:** repeated opening/closing converges in finite steps (idempotency)
- **Monotone reward shaping:** if $A \subseteq B$ then applying the same morphological operation maintains the subset relationship — the agent's actions have predictable ordering effects
- **Compositional planning:** the agent can reason about sequences of operations algebraically

## Morphological Reconstruction — Geodesic Operations

Morphological reconstruction is a powerful operation that extracts connected components of objects based on a marker image, without distorting the shapes.

### Step 1: Geodesic Dilation

**Definition:** The geodesic dilation of marker $M$ with respect to mask $G$ is:

$$\delta_G^{(1)}(M) = (M \oplus B) \cap G$$

where $B$ is the structuring element. The mask $G$ constrains the dilation: it can never exceed $G$.

**Iterated geodesic dilation:**
$$\delta_G^{(n)}(M) = \delta_G^{(1)}(\delta_G^{(n-1)}(M))$$

### Step 2: Reconstruction by Dilation

$$R_G^{\delta}(M) = \delta_G^{(\infty)}(M) = \lim_{n \to \infty} \delta_G^{(n)}(M)$$

**Theorem:** The sequence $\{\delta_G^{(n)}(M)\}_{n=0}^{\infty}$ converges in a finite number of iterations.

**Proof:** 
1. $\delta_G^{(n)}(M) \subseteq G$ for all $n$ (bounded by mask)
2. $\delta_G^{(n)}(M) \subseteq \delta_G^{(n+1)}(M)$ (monotonically increasing — dilation only adds pixels)
3. $G$ has finitely many pixels

A monotonically increasing sequence bounded above in a finite set must converge. The number of iterations is at most the diameter of the largest connected component of $G$. $\blacksquare$

### Step 3: Properties of Reconstruction

1. **Idempotent:** $R_G^\delta(R_G^\delta(M)) = R_G^\delta(M)$ (applying reconstruction twice gives the same result)
2. **Increasing:** If $M_1 \subseteq M_2$, then $R_G^\delta(M_1) \subseteq R_G^\delta(M_2)$
3. **Anti-extensive:** $R_G^\delta(M) \subseteq G$ (output never exceeds mask)
4. **Absorbs connected components:** $R_G^\delta(M)$ contains exactly the connected components of $G$ that are "touched" by marker $M$

### Step 4: Applications

**Filling holes:** To fill holes in binary image $A$:
- Mask: $G = A^c$ (complement)
- Marker: $M = $ border pixels of $A^c$
- Holes = $G \setminus R_G^\delta(M)$ (complement of reconstruction = holes)

**Opening by reconstruction** (shape-preserving opening):
$$\gamma_{\text{rec}}(A) = R_A^\delta(A \ominus B)$$

Unlike standard opening, this preserves the complete shape of objects that survive erosion, rather than distorting them.

### Step 5: RL Application — Marker-Based Segmentation

An RL agent can learn to place markers (seeds) on an image, then use reconstruction to extract complete objects:
- **State:** Image + current marker positions
- **Actions:** Place marker at $(x, y)$, remove marker, change marker intensity
- **Transition:** Apply reconstruction to get segmentation
- **Reward:** IoU with ground truth segmentation

This formulation allows the agent to learn an interactive segmentation strategy where each marker placement improves the result.

## 2. RL Formulation: Morphological Agent

| Component | Definition |
|:----------|:-----------|
| State $s_t$ | Current binary/grayscale image |
| Actions | {erode_3, erode_5, dilate_3, dilate_5, open, close, gradient, identity} |
| Reward | $r = \text{IoU}(I_{t+1}, I_{target})$ |
| Goal | Learn sequence of morph ops to clean noisy binary images |

---
**Next:** Module 2.4 — Image Transformations